In [2]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import quote
import os
import time


def crawl_saramin():
    results = []

    keyword = "데이터분석"
    page = 1

    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    encoded_keyword = quote(keyword)

    url = (
        "https://www.saramin.co.kr/zf_user/search/recruit"
        f"?searchType=search&searchword={encoded_keyword}&recruitPage={page}"
    )

    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    job_items = soup.select("div.item_recruit")

    for item in job_items:
        company_tag = item.select_one("div.area_corp strong.corp_name a")
        title_tag = item.select_one("div.area_job h2.job_tit a")
        detail_tags = item.select("div.area_job div.job_condition span")

        company = company_tag.get_text(strip=True) if company_tag else ""
        recruit = title_tag.get_text(strip=True) if title_tag else ""

        if title_tag and title_tag.has_attr("href"):
            href = title_tag["href"]

            if href.startswith("http"):
                url_link = href
            else:
                url_link = "https://www.saramin.co.kr" + href
        else:
            url_link = ""

        detail = ", ".join([tag.get_text(strip=True) for tag in detail_tags])

        if company or recruit:
            results.append({
                "Site": "Saramin",
                "Col_Company": company,
                "Col_Recruit": recruit,
                "Col_detail": detail,
                "Col_url": url_link
            })

    df = pd.DataFrame(
        results,
        columns=["Site", "Col_Company", "Col_Recruit", "Col_detail", "Col_url"]
    )

    return df


df_saramin = crawl_saramin()

os.makedirs("data_tmp", exist_ok=True)

df_saramin.to_csv(
    "data_tmp/data_saramin.csv",
    index=False,
    encoding="utf-8-sig"
)

print("data_tmp/data_saramin.csv 저장 완료")
print(df_saramin.shape)

df_saramin.head()


data_tmp/data_saramin.csv 저장 완료
(40, 5)


,Site,Col_Company,Col_Recruit,Col_detail,Col_url
0,Saramin,(주)디케이비엠시,[DKBMC] Tableau 및데이터분석가 채용(신입/경력),"서울강남구, 신입·경력, 초대졸↑, 정규직·계약직",https://www.saramin.co.kr/zf_user/jobs/relay/v...
1,Saramin,(주)바른인사컨설팅,[외국계/복리후생 上]Data Support specialist (데이터분석/KPI),"서울강남구, 경력무관, 대졸↑, 파견직 (정규직 전환가능)·파견직",https://www.saramin.co.kr/zf_user/jobs/relay/v...
2,Saramin,넛지헬스케어(주),[캐시워크]데이터분석담당 채용전환형 인턴,"서울강남구, 신입, 대졸↑, 인턴직",https://www.saramin.co.kr/zf_user/jobs/relay/v...
3,Saramin,(주)에치와이,(주)에치와이(한국야쿠르트)데이터분석가 경력 채용,"서울서초구, 경력5년↑, 대졸↑, 정규직",https://www.saramin.co.kr/zf_user/jobs/relay/v...
4,Saramin,(주)하쿠호도제일,[하쿠호도제일] AI데이터분석팀 경력직 모집,"서울마포구, 경력 2~5년, 대졸↑, 계약직",https://www.saramin.co.kr/zf_user/jobs/relay/v...
